In [1]:
import pandas as pd


df = pd.read_csv(r"C:\Users\Madhubala\Downloads\fish_safety_4000_balanced_discrete_waterlevel.csv")
print("Existing dataset loaded successfully!")
df.tail()


Existing dataset loaded successfully!


,FISH_NAME,PH,TEMPERATURE,LIGHT_INTENSITY,AIR_QUALITY,HUMIDITY,WATER_LEVEL,SAFE_STATUS
4001,ANTARCTIC ICEFISH,6.9,1.5,12.0,38.0,68.0,5,SAFE
4002,ANTARCTIC ICEFISH,7.5,3.0,20.0,45.0,72.0,6,SAFE
4003,ANTARCTIC ICEFISH,7.9,8.0,45.0,85.0,70.0,4,NOT SAFE
4004,ANTARCTIC ICEFISH,6.2,10.0,60.0,90.0,75.0,3,NOT SAFE
4005,ANTARCTIC ICEFISH,7.4,20.0,70.0,80.0,72.0,5,NOT SAFE


In [2]:
print(df['SAFE_STATUS'].value_counts())


SAFE_STATUS
SAFE        2003
NOT SAFE    2003
Name: count, dtype: int64


In [3]:
# 🐟 Fish Safety Training - Fish-Aware Version

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from joblib import dump



df.columns = df.columns.str.strip()
print(f"✅ Loaded {len(df)} samples")

 
fish_encoder = LabelEncoder()
df['FISH_NAME_ENCODED'] = fish_encoder.fit_transform(df['FISH_NAME'])

status_encoder = LabelEncoder()
df['SAFE_STATUS_ENCODED'] = status_encoder.fit_transform(df['SAFE_STATUS'])


X = df[['FISH_NAME_ENCODED', 'PH', 'TEMPERATURE', 'LIGHT_INTENSITY', 'AIR_QUALITY', 'HUMIDITY', 'WATER_LEVEL']]
y = df['SAFE_STATUS_ENCODED']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"🎯 Model Accuracy: {acc*100:.2f}%")


dump(model, "fish_safety_model.joblib")
dump(fish_encoder, "fish_name_encoder.joblib")
dump(status_encoder, "status_encoder.joblib")

print("✅ Model & encoders saved successfully!")


✅ Loaded 4006 samples
🎯 Model Accuracy: 100.00%
✅ Model & encoders saved successfully!


In [5]:
import serial
import time
import re
import pandas as pd
from joblib import load
import os


print("📦 Loading ML model...")
model = load("fish_safety_model.joblib")
fish_encoder = load("fish_name_encoder.joblib")
print("✅ Model and encoders loaded successfully!\n")

# --------------------------------------------
# 2️⃣ Connect to Arduino
# --------------------------------------------
port = "COM4"   # 🔁 Change this if needed
baud = 9600
print(f"🔌 Connecting to Arduino on {port}...")
ser = serial.Serial(port, baud, timeout=1)
time.sleep(2)
print("✅ Connected to Arduino successfully!\n")

# --------------------------------------------
# 3️⃣ Fish Name Input
# --------------------------------------------
fish_name = input("🐠 Enter the fish name: ").strip().upper()

try:
    fish_encoded = fish_encoder.transform([fish_name])[0]
except Exception:
    print(f"⚠️ Fish name '{fish_name}' not recognized by encoder. Using default value 0.")
    fish_encoded = 0

print("\n📡 Reading live data from Arduino...\n")

# --------------------------------------------
# 4️⃣ Initialize Variables
# --------------------------------------------
temperature = None
humidity = None
air_quality = None
ph_value = None
light_intensity = None
water_level = None
reading_count = 0
MAX_READINGS = 6  # ✅ Automatically stop after 6 readings

# --------------------------------------------
# 5️⃣ Live Data Reading Loop
# --------------------------------------------
try:
    while reading_count < MAX_READINGS:
        line = ser.readline().decode('utf-8', errors='ignore').strip()
        if not line or line.strip('-').strip() == '':
            continue

        print(f"📥 Raw: {line}")

        # 🌡️ Temperature + Humidity
        if "Temperature" in line and "Humidity" in line:
            temp_match = re.search(r"Temperature:\s*([\d\.]+)", line)
            hum_match = re.search(r"Humidity:\s*([\d\.]+)", line)
            if temp_match:
                temperature = float(temp_match.group(1))
            if hum_match:
                humidity = float(hum_match.group(1))

        # 🌫️ Air Quality
        if "Air Quality" in line:
            match = re.search(r"Air Quality.*?:\s*(\d+)", line)
            if match:
                air_quality = float(match.group(1))

        # ⚗️ pH Sensor + Light + Water Level
        if "PH:" in line:
            ph_match = re.search(r"PH:(\d+\.?\d*)", line)
            l_match = re.search(r"L:\s*(\d+)", line)
            w_match = re.search(r"W:\s*(\d+)", line)  # extract WATER LEVEL
            
            if ph_match:
                ph_value = float(ph_match.group(1))
            if l_match:
                light_intensity = float(l_match.group(1))
            if w_match:
                water_level = float(w_match.group(1))

        # --------------------------------------------
        # 6️⃣ When all readings available → Adjust + Predict
        # --------------------------------------------
        if all(v is not None for v in [ph_value, temperature, humidity, air_quality, light_intensity, water_level]):
            
            # ✅ Adjust the raw sensor values for more realistic underwater data
            # temperature = temperature - 4            # Approx water temp (air temp - 4°C)
            air_quality = max(0, air_quality - 50)   # Simulate underwater air quality

            # # ✅ Adjust high pH (natural buffering effect)
            if ph_value > 8.0:
                ph_value = ph_value - 0.4

            # 🧠 Predict with 7 features
            features = [[fish_encoded, ph_value, temperature, light_intensity, air_quality, humidity, water_level]]
            prediction = model.predict(features)[0]
            status = "SAFE" if prediction == 1 else "NOT SAFE"

            # --------------------------------------------
            # 🔁 Send SAFE / NOTSAFE signal to Arduino
            # --------------------------------------------
            if status == "SAFE":
                ser.write(b'SAFE\n')
            else:
                ser.write(b'NOTSAFE\n')

            # --------------------------------------------
            # 🧾 Print Reading Summary
            # --------------------------------------------
            reading_count += 1
            print(f"\n📘 Reading #{reading_count}")
            print(f"🐟 Fish: {fish_name}")
            print(f"📊 pH={ph_value:.2f}, Temp≈{temperature:.2f}°C, Humidity={humidity:.2f}%, Air={air_quality:.2f}, Light={light_intensity:.2f}, Water={water_level:.2f}")
            print(f"🔮 Prediction: {'✅ SAFE' if status == 'SAFE' else '⚠️ NOT SAFE'}\n")

            # Reset for next reading
            ph_value = temperature = humidity = air_quality = light_intensity = water_level = None

        time.sleep(0.2)

    # ✅ Stop automatically after MAX_READINGS
    print(f"\n✅ Collected {MAX_READINGS} readings successfully. Stopping...\n")

except KeyboardInterrupt:
    print("\n🛑 Stopped by user.")

finally:
    ser.close()
    print("🔒 Serial connection closed safely.")


📦 Loading ML model...
✅ Model and encoders loaded successfully!

🔌 Connecting to Arduino on COM4...
✅ Connected to Arduino successfully!



🐠 Enter the fish name:  GOLDFISH



📡 Reading live data from Arduino...

📥 Raw: =================================
📥 Raw: 💧 Smart Fish Tank Monitoring System
📥 Raw: Sensors: DHT11 | pH | Air Quality | LEDs
📥 Raw: =================================
📥 Raw: ⚠️ DHT11 sensor read failed!
📥 Raw: 🌫️ Air Quality (Analog): 259
📥 Raw: ⚗️ pH Sensor Raw Data: PH:37.65, W: 0, L:   0, T: 110,
📥 Raw: ⚠️ DHT11 sensor read failed!
📥 Raw: 🌫️ Air Quality (Analog): 236
📥 Raw: ⚗️ pH Sensor Raw Data: PH:37.65, W: 0, L:   0, T: 110,PH:37.65, W: 0, L:   0, T: 1
📥 Raw: ⚠️ DHT11 sensor read failed!
📥 Raw: 🌫️ Air Quality (Analog): 255
📥 Raw: ⚗️ pH Sensor Raw Data: PH:37.65, W: 0, L:   0, T: 110,PH:37.65, W: 0, L:   0, T: 1
📥 Raw: ⚠️ DHT11 sensor read failed!
📥 Raw: 🌫️ Air Quality (Analog): 251
📥 Raw: ⚗️ pH Sensor Raw Data: PH:37.69, W: 0, L:   0, T: 108,
📥 Raw: ⚠️ DHT11 sensor read failed!
📥 Raw: 🌫️ Air Quality (Analog): 253
📥 Raw: ⚗️ pH Sensor Raw Data: PH:37.65, W: 0, L:   0, T: 108,PH:37.65, W: 0, L:   0, T: 1
📥 Raw: ⚠️ DHT11 sensor read failed!

SerialException: ClearCommError failed (PermissionError(13, 'The device does not recognize the command.', None, 22))